In [1]:
import pandas as pd
df = pd.read_parquet("../classify_data.parquet").reset_index()

In [2]:
class_map = {
    '0': 0,
    '1-3': 1,
    '4-7': 2,
    '8-14': 3,
    '15-30': 4,
    '31-60': 5
}
df['resolution_class_int'] = df['resolution_class'].map(class_map)

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

feature_columns = ['Agency','Complaint Type','Descriptor','Location Type', 'Incident Zip', 'City', 'Borough',
                    'Open Data Channel Type', 'Latitude', 'Longitude',
                    'hour_of_day', 'day_of_week', 'month', 'is_weekend', 'open_requests_past_24h']

X = df[feature_columns]
y = df['resolution_class_int']

#Segregate columsn for preprocessing
categorical_cols = ['Agency','Complaint Type','Descriptor','Location Type', 'Incident Zip', 'City', 'Borough',
                    'Open Data Channel Type' ,'hour_of_day','day_of_week', 'month', 'is_weekend']
numeric_cols = ['Latitude','Longitude','open_requests_past_24h']

# Preprocessing -> Normalization + encoding categories
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numeric_cols)
    ]
)

lr = LogisticRegression(
    solver='saga',     
    max_iter=200,      
    tol=0.005,
    class_weight='balanced',
    n_jobs=8,         
    verbose=3         
)

#Full pipeline: preprocessing + model
model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('clf', lr)
])

#Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [4]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1_macro')

print(scores)

# 6. Fit model
model.fit(X_train, y_train)

# 7. Predict
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

Epoch 15, change: 0.01193875
Epoch 16, change: 0.011693167
Epoch 17, change: 0.010527612
Epoch 18, change: 0.010508953
Epoch 19, change: 0.009619534
Epoch 20, change: 0.0093873626
Epoch 21, change: 0.0090874788
Epoch 22, change: 0.0084148251
Epoch 23, change: 0.0082649933
Epoch 24, change: 0.0075777771
Epoch 25, change: 0.0072234939
Epoch 26, change: 0.0068491589
Epoch 27, change: 0.0064922065
Epoch 28, change: 0.0061393868
Epoch 29, change: 0.0057108675
Epoch 30, change: 0.0055082625
Epoch 31, change: 0.0053046546
convergence after 32 epochs took 35 seconds
[0.43631707 0.43232797 0.43479296]


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.11304927
Epoch 3, change: 0.06246565
Epoch 4, change: 0.076280361
Epoch 5, change: 0.042411871
Epoch 6, change: 0.03741608
Epoch 7, change: 0.026458586
Epoch 8, change: 0.023499006
Epoch 9, change: 0.020520387
Epoch 10, change: 0.01769806
Epoch 11, change: 0.017543085
Epoch 12, change: 0.014177103
Epoch 13, change: 0.013797897
Epoch 14, change: 0.012329564
Epoch 15, change: 0.011752076
Epoch 16, change: 0.011082502
Epoch 17, change: 0.010741019
Epoch 18, change: 0.010288804
Epoch 19, change: 0.0094836245
Epoch 20, change: 0.0090969265
Epoch 21, change: 0.008531751
Epoch 22, change: 0.0081610216
Epoch 23, change: 0.0078197649
Epoch 24, change: 0.0072575368
Epoch 25, change: 0.0069548061
Epoch 26, change: 0.0064364702
Epoch 27, change: 0.0058230972
Epoch 28, change: 0.0056523888
Epoch 29, change: 0.0054286107
Epoch 30, change: 0.0052412082
Epoch 31, change: 0.0051144468
convergence after 32 epochs took 55 seconds


In [5]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Test Accuracy: 0.7276755968469885

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.84      0.90    331345
           1       0.52      0.57      0.55     76042
           2       0.26      0.34      0.30     27143
           3       0.23      0.28      0.25     17086
           4       0.20      0.36      0.26     15160
           5       0.24      0.61      0.35     10482

    accuracy                           0.73    477258
   macro avg       0.41      0.50      0.43    477258
weighted avg       0.80      0.73      0.76    477258


Confusion Matrix:
 [[277576  33531   7971   3443   4467   4357]
 [  4621  43693  12327   5000   5976   4425]
 [   570   4677   9245   4323   4883   3445]
 [   251   1131   3166   4861   4491   3186]
 [   130    543   1820   2645   5523   4499]
 [    68    134    568    827   2494   6391]]


In [6]:
import numpy as np
np.save("../results/regresssion_predictions.npy", y_pred)
np.save("../results/regresssion_prob.npy", y_prob)

In [ ]:
y_pred = np.load("../results/regresssion_predictions.npy")

NameError: name 'np' is not defined

In [ ]:
print(X.shape)
print(y.shape)
df.dtypes